# ANN — Fashion MNIST
Fully-connected classifier on Fashion MNIST (10 classes, 28×28 grayscale images flattened to 784 features).

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Data
`fashion_mnist.csv` — first column is label, remaining 784 are pixel values (0–255).

In [ ]:
CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
                'Sandal','Shirt','Sneaker','Bag','Ankle boot']

df = pd.read_csv('data/fashion_mnist.csv')
print(df.shape)
df.head()

## EDA — Sample Images

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("Fashion MNIST — Sample Images", fontsize=14)
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(CLASS_NAMES[df.iloc[i, 0]], fontsize=9)
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
X = df.iloc[:, 1:].values / 255.0
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Dataset & DataLoader

In [ ]:
class FashionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_loader = DataLoader(FashionDataset(X_train, y_train), batch_size=32, shuffle=True, pin_memory=True)
test_loader  = DataLoader(FashionDataset(X_test,  y_test),  batch_size=32, shuffle=False, pin_memory=True)
print(f"Train batches: {len(train_loader)}")

## Model

In [ ]:
class ANN(nn.Module):
    def __init__(self, num_features, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

## Training

In [ ]:
epochs = 30
model = ANN(X_train.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg = total_loss / len(train_loader)
    train_losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}  Loss: {avg:.4f}")

## Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), train_losses, marker='o', markersize=3)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
def evaluate(model, loader, name=""):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            _, predicted = torch.max(model(features), 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    acc = correct / total * 100
    print(f"{name} Accuracy: {acc:.2f}%")
    return acc

evaluate(model, train_loader, "Train")
evaluate(model, test_loader,  "Test")

## Per-Class Accuracy

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        _, preds = torch.max(model(features), 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Save Model

In [ ]:
torch.save(model.state_dict(), 'ann_fashion_mnist.pth')
print("Model saved.")